In [2]:
import sys
sys.path.insert(0, '../..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.data.storage.database import get_engine

plt.style.use('seaborn-v0_8-whitegrid')
engine = get_engine()

with engine.connect() as conn:
    prices = pd.read_sql("""
        SELECT p.date, p.close, p.adjusted_close, p.volume,
               p.week52_low, p.week52_high, s.ticker, s.sector
        FROM prices p
        JOIN securities s ON p.security_id = s.security_id
        WHERE s.security_type = 'equity'
        AND p.close IS NOT NULL
        AND p.week52_low IS NOT NULL
        AND p.week52_high IS NOT NULL
    """, conn)

    returns = pd.read_sql("""
        SELECT r.date, r.adj_daily_return, s.ticker
        FROM returns r
        JOIN securities s ON r.security_id = s.security_id
        WHERE s.security_type = 'equity'
        AND r.adj_daily_return IS NOT NULL
        AND ABS(r.adj_daily_return) <= 1.0
    """, conn)

prices['date'] = pd.to_datetime(prices['date'])
returns['date'] = pd.to_datetime(returns['date'])

print(f"Prices: {len(prices):,} rows with 52-week range data")
print(f"Returns: {len(returns):,} rows")

Prices: 280,718 rows with 52-week range data
Returns: 249,488 rows


In [2]:
# Factor 1: 52-week low proximity
# How close is current price to 52-week low?
# Formula: (price - 52w_low) / (52w_high - 52w_low)
# 0 = at 52-week low (cheapest) 
# 1 = at 52-week high (most expensive)
# Low values = value signal (stock is cheap relative to recent range)

prices['range'] = prices['week52_high'] - prices['week52_low']
prices['proximity'] = np.where(
    prices['range'] > 0,
    (prices['close'] - prices['week52_low']) / prices['range'],
    np.nan
)

# Sanity check
valid = prices['proximity'].dropna()
print(f"Valid proximity values: {len(valid):,}")
print(f"Range: {valid.min():.3f} to {valid.max():.3f}")
print(f"Mean:  {valid.mean():.3f}")
print(f"Values outside [0,1]: {((valid < 0) | (valid > 1)).sum()}")
print(f"\nSample:")
print(prices[['ticker','date','close','week52_low','week52_high','proximity']]
      .dropna(subset=['proximity']).head(8).to_string(index=False))

Valid proximity values: 262,571
Range: -26.429 to 507.857
Mean:  2.717
Values outside [0,1]: 115233

Sample:
ticker       date  close  week52_low  week52_high  proximity
  EGAD 2007-01-02  52.00        22.0        57.00   0.857143
  KAPC 2007-01-02 100.00       111.0       148.00  -0.297297
  KUKZ 2007-01-02  43.50        67.5        89.00  -1.116279
   REA 2007-01-02  25.50        14.5        23.50   1.222222
  SASN 2007-01-02 140.00        10.5        13.60  41.774194
   WTK 2007-01-02 120.00       180.0       290.00  -0.545455
   C&G 2007-01-02  50.00        21.0        29.00   3.625000
  FIRE 2007-01-02  24.75         3.4         5.95   8.372549


In [3]:
# Factor: 6-month price momentum (long horizon)
# Stocks with worst 6-month returns = value-like signal
# 126 trading days ≈ 6 months

print("Computing 6-month and 12-month momentum factors...")

long_mom = []

for ticker, group in prices.groupby('ticker'):
    g = group.sort_values('date').copy()
    adj = pd.to_numeric(g['adjusted_close'], errors='coerce')
    log_prices = np.log(adj)
    
    # 6-month momentum (skip 1 month = 21 days)
    mom_6m = log_prices.shift(21) - log_prices.shift(126 + 21)
    
    # 12-month momentum (skip 1 month)
    mom_12m = log_prices.shift(21) - log_prices.shift(252 + 21)
    
    long_mom.append(pd.DataFrame({
        'ticker': ticker,
        'date': g['date'].values,
        'mom_6m': mom_6m.values,
        'mom_12m': mom_12m.values,
    }))

long_mom_df = pd.concat(long_mom, ignore_index=True)

valid_6m = long_mom_df['mom_6m'].notna().sum()
valid_12m = long_mom_df['mom_12m'].notna().sum()
print(f"Valid 6m observations:  {valid_6m:,}")
print(f"Valid 12m observations: {valid_12m:,}")
print(f"\nSample:")
print(long_mom_df.dropna(subset=['mom_6m']).head(5).to_string(index=False))

Computing 6-month and 12-month momentum factors...
Valid 6m observations:  268,387
Valid 12m observations: 257,994

Sample:
ticker       date   mom_6m  mom_12m
  ABSA 2013-08-02 0.018928      NaN
  ABSA 2013-08-05 0.015699      NaN
  ABSA 2013-08-06 0.049546      NaN
  ABSA 2013-08-07 0.070690      NaN
  ABSA 2013-08-08 0.079553      NaN


In [4]:
# Quintile analysis — long horizon momentum
merged = long_mom_df.merge(
    returns[['ticker','date','adj_daily_return']],
    on=['ticker','date']
).dropna(subset=['mom_6m'])

merged = merged.sort_values(['ticker','date'])
merged['fwd_return'] = merged.groupby('ticker')['adj_daily_return'].shift(-1)
merged = merged.dropna(subset=['fwd_return'])

print("=== LONG-HORIZON MOMENTUM QUINTILES ===\n")
print(f"{'Factor':<10} {'Q1 (losers)':<14} {'Q5 (winners)':<14} {'L/S Spread':<12} {'Q1 t-stat'}")
print("-" * 60)

for col in ['mom_6m', 'mom_12m']:
    data = merged.dropna(subset=[col])
    
    quintile_rets = {q: [] for q in [1, 5]}
    
    for date, group in data.groupby('date'):
        if len(group) < 15:
            continue
        try:
            group = group.copy()
            group['q'] = pd.qcut(
                group[col], q=5, labels=[1,2,3,4,5], duplicates='drop'
            )
            for q in [1, 5]:
                q_ret = group[group['q']==q]['fwd_return']
                if len(q_ret) >= 3:
                    quintile_rets[q].append(q_ret.mean())
        except Exception:
            continue
    
    q1 = pd.Series(quintile_rets[1]).dropna()
    q5 = pd.Series(quintile_rets[5]).dropna()
    
    q1_ann = q1.mean() * 252 * 100
    q5_ann = q5.mean() * 252 * 100
    spread = q5_ann - q1_ann
    t1 = q1.mean() / (q1.std() / np.sqrt(len(q1)))
    
    print(f"{col:<10} {q1_ann:>+10.1f}%   {q5_ann:>+10.1f}%   "
          f"{spread:>+8.1f}%   {t1:>+6.2f}")

print("\nComparison with short-horizon (20d) factor:")
print(f"{'mom_20d':<10} {'  +83.2%':<14} {'  -71.3%':<14} {' -154.5%':<12} {' +5.51'}")

=== LONG-HORIZON MOMENTUM QUINTILES ===

Factor     Q1 (losers)    Q5 (winners)   L/S Spread   Q1 t-stat
------------------------------------------------------------
mom_6m           -7.9%         -0.0%       +7.8%    -0.53
mom_12m         -12.5%         +8.4%      +20.9%    -0.83

Comparison with short-horizon (20d) factor:
mom_20d      +83.2%         -71.3%        -154.5%      +5.51


In [5]:
print("""
=== FINDING — LONG HORIZON VALUE PROXIES ===

6-month momentum:  Q1 return -7.9%,  t-stat -0.53  NOT significant
12-month momentum: Q1 return -12.5%, t-stat -0.83  NOT significant

CONCLUSION:
Long-horizon losers do NOT outperform in NSE.
They continue underperforming — genuine fundamental deterioration.

The mean reversion effect is EXCLUSIVELY short-horizon (5-20 days).
Beyond 20 days, the signal reverses.

This eliminates long-horizon value proxies as factors.
True value factors require fundamental data:
  - Price-to-book ratio
  - Dividend yield  
  - Earnings yield

These require ingesting the individual stock files.
Added to research backlog.

CURRENT FACTOR RANKING:
  1. 20d mean reversion — CONFIRMED, t-stat 5.51
  2. 6m/12m value proxies — REJECTED, not significant
""")


=== FINDING — LONG HORIZON VALUE PROXIES ===

6-month momentum:  Q1 return -7.9%,  t-stat -0.53  NOT significant
12-month momentum: Q1 return -12.5%, t-stat -0.83  NOT significant

CONCLUSION:
Long-horizon losers do NOT outperform in NSE.
They continue underperforming — genuine fundamental deterioration.

The mean reversion effect is EXCLUSIVELY short-horizon (5-20 days).
Beyond 20 days, the signal reverses.

This eliminates long-horizon value proxies as factors.
True value factors require fundamental data:
  - Price-to-book ratio
  - Dividend yield  
  - Earnings yield

These require ingesting the individual stock files.
Added to research backlog.

CURRENT FACTOR RANKING:
  1. 20d mean reversion — CONFIRMED, t-stat 5.51
  2. 6m/12m value proxies — REJECTED, not significant

